In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("✓ Cell 1 complete")

Torch version: 2.9.1+cpu
CUDA available: False
✓ Cell 1 complete


In [2]:
import os
import sys

yolov5_path = os.path.abspath('yolov5')
print("YOLOv5 path:", yolov5_path)
print("YOLOv5 exists:", os.path.exists(yolov5_path))

if os.path.exists(yolov5_path):
    sys.path.insert(0, yolov5_path)
    print("✓ Path added")
print("✓ Cell 2 complete")

YOLOv5 path: c:\Research\Thermal-Dataset\ExplainableAI\yolov5
YOLOv5 exists: True
✓ Path added
✓ Cell 2 complete


In [3]:
from models.common import DetectMultiBackend

weights_path = r"C:\Research\Thermal-Dataset\ExplainableAI\weights\yolov5_lar\best.pt"

print("Loading model...")
try:
    model = DetectMultiBackend(weights_path, device='cpu', fp16=False)
    print("✓ Model loaded successfully!")
    print("Model type:", type(model))
    print("Has names:", hasattr(model, 'names'))
    if hasattr(model, 'names'):
        print("Classes:", model.names)
except Exception as e:
    print("✗ Error:", e)
    import traceback
    traceback.print_exc()

print("✓ Cell 3 complete")

Loading model...


Fusing layers... 
YOLOv5l summary: 392 layers, 46604156 parameters, 0 gradients


✓ Model loaded successfully!
Model type: <class 'models.common.DetectMultiBackend'>
Has names: True
Classes: {0: 'handgun', 1: 'knife', 2: 'person', 3: 'revolver', 4: 'rifle'}
✓ Cell 3 complete


In [4]:
import cv2
import numpy as np
import glob

# UPDATE THIS: Path to folder containing test images
test_folder = r"C:\Research\Thermal-Dataset\ExplainableAI\weights\test"

# UPDATE THIS: Output folder for XAI results
output_folder = r"C:\Research\Thermal-Dataset\ExplainableAI\weights\yolov5_lar\xai_output"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get list of image files (supports jpg, jpeg, png)
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(os.path.join(test_folder, ext)))

print(f"Found {len(image_files)} images in {test_folder}")
print("Output folder:", output_folder)
print("✓ Cell 4 complete")

Found 20 images in C:\Research\Thermal-Dataset\ExplainableAI\weights\test
Output folder: C:\Research\Thermal-Dataset\ExplainableAI\weights\yolov5_lar\xai_output
✓ Cell 4 complete


In [5]:
from utils.general import non_max_suppression

def generate_gradcam(model, img_tensor, detections):
    """
    Generate Grad-CAM heatmap for explainability (FIXED VERSION)
    
    Returns:
        cam: Numpy array with attention heatmap (values 0-1)
    """
    print("\n" + "="*60)
    print("GENERATING GRAD-CAM (Explainable AI)")
    print("="*60)
    
    if len(detections) == 0:
        print("⚠ No detections found - cannot generate Grad-CAM")
        return None
    
    # Step 1: Find the target layer
    print("\n1. Finding target layer...")
    conv_layers = []
    for name, module in model.model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            conv_layers.append((name, module))
    
    if len(conv_layers) == 0:
        print("✗ No conv layers found")
        return None
    
    target_layer = conv_layers[-4][1] if len(conv_layers) > 3 else conv_layers[-1][1]
    print(f"✓ Using layer: {conv_layers[-4][0] if len(conv_layers) > 3 else conv_layers[-1][0]}")
    
    # Step 2: Storage for activations and gradients (moved to CPU immediately)
    print("\n2. Setting up hooks...")
    activations = []
    gradients = []
    
    def forward_hook(module, input, output):
        """Captures feature maps - clone and move to CPU immediately"""
        # Clone to avoid view issues, detach to break computational graph, move to CPU
        activations.append(output.clone().detach().cpu())
    
    def backward_hook(module, grad_input, grad_output):
        """Captures gradients - clone and move to CPU immediately"""
        if grad_output[0] is not None:
            # Clone to avoid view issues, detach, and move to CPU
            gradients.append(grad_output[0].clone().detach().cpu())
    
    # Register hooks
    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)
    print("✓ Hooks registered")
    
    try:
        # Step 3: Forward pass - create completely new tensor
        print("\n3. Forward pass with gradients...")
        model.zero_grad()
        
        # Create a fresh copy of the input that requires gradients
        img_input = img_tensor.detach().clone()
        img_input.requires_grad_(True)
        
        # Forward pass
        outputs = model.model(img_input)
        
        # Handle different output formats
        if isinstance(outputs, (list, tuple)):
            outputs = outputs[0]
        
        # Step 4: Get the target score
        if outputs.shape[0] > 0 and outputs.shape[1] >= 5:
            confidence_scores = outputs[:, 4]
            max_score = confidence_scores.max()
            print(f"✓ Target detection confidence: {max_score:.3f}")
            
            # Step 5: Backward pass
            print("\n4. Computing gradients (backward pass)...")
            
            # Zero gradients before backward
            if img_input.grad is not None:
                img_input.grad.zero_()
            
            # Backward pass - DO NOT use retain_graph
            max_score.backward(retain_graph=False)
            
            print("✓ Gradients computed")
            
        else:
            print("✗ Invalid output format")
            h1.remove()
            h2.remove()
            return None
            
    except RuntimeError as e:
        if "view" in str(e) or "inplace" in str(e):
            print(f"⚠ Caught view/inplace error, trying alternative method...")
            h1.remove()
            h2.remove()
            return generate_gradcam_alternative(model, img_tensor, detections)
        else:
            print(f"✗ Runtime error: {e}")
            h1.remove()
            h2.remove()
            return None
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()
        h1.remove()
        h2.remove()
        return None
    
    # Step 6: Generate CAM
    print("\n5. Generating Class Activation Map...")
    
    # Remove hooks before processing
    h1.remove()
    h2.remove()
    
    try:
        if len(gradients) > 0 and len(activations) > 0:
            # Everything is already on CPU, convert to numpy
            grads = gradients[0].numpy()[0]  # Shape: [channels, H, W]
            acts = activations[0].numpy()[0]  # Shape: [channels, H, W]
            
            print(f"   Gradients shape: {grads.shape}")
            print(f"   Activations shape: {acts.shape}")
            
            # Global average pooling on gradients
            weights = np.mean(grads, axis=(1, 2))
            print(f"   Computed {len(weights)} channel weights")
            
            # Weighted combination of activation maps
            cam = np.zeros(acts.shape[1:], dtype=np.float32)
            for i, w in enumerate(weights):
                cam += w * acts[i]
            
            # Apply ReLU
            cam = np.maximum(cam, 0)
            
            # Normalize
            if cam.max() > 0:
                cam = cam / cam.max()
            
            print(f"✓ CAM generated: {cam.shape}, range [{cam.min():.3f}, {cam.max():.3f}]")
            return cam
        else:
            print(f"✗ No gradients captured (grads: {len(gradients)}, acts: {len(activations)})")
            return None
            
    except Exception as e:
        print(f"✗ Error creating CAM: {e}")
        import traceback
        traceback.print_exc()
        return None


def generate_gradcam_alternative(model, img_tensor, detections):
    """
    Alternative Grad-CAM method using input gradients
    """
    print("\n" + "="*60)
    print("TRYING ALTERNATIVE GRAD-CAM METHOD")
    print("="*60)
    
    try:
        # Find conv layer
        conv_layers = []
        for name, module in model.model.named_modules():
            if isinstance(module, torch.nn.Conv2d):
                conv_layers.append((name, module))
        
        if len(conv_layers) == 0:
            return None
        
        target_layer = conv_layers[-4][1] if len(conv_layers) > 3 else conv_layers[-1][1]
        
        # Simpler approach - just capture activation
        activation = None
        
        def hook(module, input, output):
            nonlocal activation
            activation = output.clone().detach()
        
        handle = target_layer.register_forward_hook(hook)
        
        # Forward pass only
        model.zero_grad()
        with torch.no_grad():
            _ = model.model(img_tensor)
        
        handle.remove()
        
        if activation is not None:
            # Use activation magnitude as proxy for importance
            act_np = activation.cpu().numpy()[0]
            
            # Simple average across channels
            cam = np.mean(np.abs(act_np), axis=0)
            
            # Normalize
            cam = np.maximum(cam, 0)
            if cam.max() > 0:
                cam = cam / cam.max()
            
            print("✓ Alternative CAM generated (activation-based)")
            return cam
        else:
            return None
            
    except Exception as e:
        print(f"✗ Alternative method also failed: {e}")
        return None


print("✓ Grad-CAM functions defined (with fallback)")

✓ Grad-CAM functions defined (with fallback)


In [6]:
# Process each image in the folder
print("\n" + "="*80)
print(f"PROCESSING {len(image_files)} IMAGES")
print("="*80)

for idx, image_path in enumerate(image_files, 1):
    print(f"\n{'='*80}")
    print(f"IMAGE {idx}/{len(image_files)}: {os.path.basename(image_path)}")
    print("="*80)
    
    try:
        # Load image
        print("Loading image...")
        img = cv2.imread(image_path)
        if img is None:
            print(f"✗ Failed to load image: {image_path}")
            continue
        
        print(f"✓ Image loaded: {img.shape}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (640, 640))
        print(f"✓ Image resized: {img_resized.shape}")
        
        # Convert to tensor
        print("Converting to tensor...")
        img_tensor = torch.from_numpy(img_resized).float().permute(2, 0, 1) / 255.0
        img_tensor = img_tensor.unsqueeze(0)
        print(f"Tensor shape: {img_tensor.shape}")
        
        # Run detection
        print("Running detection...")
        with torch.no_grad():
            pred = model(img_tensor)
        print("✓ Model inference complete")
        
        pred = non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
        print("✓ NMS complete")
        
        detections = pred[0].cpu().numpy()
        print(f"✓ Found {len(detections)} detections")
        
        # Generate Grad-CAM
        cam = generate_gradcam(model, img_tensor, detections)
        
        if cam is not None:
            print(f"\n✓ Grad-CAM successfully generated!")
            print(f"Heatmap shape: {cam.shape}")
            print(f"Heatmap range: [{cam.min():.3f}, {cam.max():.3f}]")
        else:
            print("\n⚠ Grad-CAM generation failed, continuing with empty heatmap")
        
        # Create visualization
        print("\nCreating Explainable AI visualization...")
        
        # Resize CAM to match image size
        if cam is not None:
            cam_resized = cv2.resize(cam, (640, 640))
            heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
            heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            print("✓ Heatmap created")
        else:
            heatmap_rgb = np.zeros_like(img_resized)
            print("⚠ Using empty heatmap")
        
        # Create 4-panel visualization
        panel_height = 640
        panel_width = 640
        
        # Panel 1: Original image
        panel1 = img_resized.copy()
        cv2.putText(panel1, '1. Original Image', (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        # Panel 2: Detections
        panel2 = img_resized.copy()
        for det in detections:
            x1, y1, x2, y2, conf, cls = int(det[0]), int(det[1]), int(det[2]), int(det[3]), det[4], int(det[5])
            cv2.rectangle(panel2, (x1, y1), (x2, y2), (0, 255, 0), 3)
            label = f'{model.names[cls]}: {conf:.2f}'
            cv2.putText(panel2, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(panel2, f'2. YOLOv5 Detections ({len(detections)})', (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        # Panel 3: Grad-CAM Heatmap
        panel3 = heatmap_rgb.copy()
        cv2.putText(panel3, '3. Grad-CAM (Model Attention)', (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(panel3, 'Red = High Attention', (10, 620), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Panel 4: Overlay (Explanation)
        if cam is not None:
            panel4 = cv2.addWeighted(img_resized, 0.5, heatmap_rgb, 0.5, 0)
            # Add bounding boxes
            for det in detections:
                x1, y1, x2, y2 = int(det[0]), int(det[1]), int(det[2]), int(det[3])
                cv2.rectangle(panel4, (x1, y1), (x2, y2), (255, 255, 255), 3)
        else:
            panel4 = img_resized.copy()
        
        cv2.putText(panel4, '4. Explanation Overlay', (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        # Combine into 2x2 grid
        top_row = np.hstack([panel1, panel2])
        bottom_row = np.hstack([panel3, panel4])
        final_visualization = np.vstack([top_row, bottom_row])
        
        # Save with unique filename based on input image
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        output_path = os.path.join(output_folder, f"{base_name}_xai.png")
        cv2.imwrite(output_path, cv2.cvtColor(final_visualization, cv2.COLOR_RGB2BGR))
        
        print(f"\n✓ Saved visualization to: {output_path}")
        print(f"✓ Image {idx}/{len(image_files)} complete!")
        
    except Exception as e:
        print(f"\n✗ Error processing {image_path}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*80)
print("BATCH PROCESSING COMPLETE!")
print(f"Processed {len(image_files)} images")
print(f"Results saved to: {output_folder}")
print("="*80)


PROCESSING 20 IMAGES

IMAGE 1/20: weapon115.jpg
Loading image...
✓ Image loaded: (640, 640, 3)
✓ Image resized: (640, 640, 3)
Converting to tensor...
Tensor shape: torch.Size([1, 3, 640, 640])
Running detection...
✓ Model inference complete
✓ NMS complete
✓ Found 1 detections

GENERATING GRAD-CAM (Explainable AI)

1. Finding target layer...
✓ Using layer: model.23.m.2.cv2.conv

2. Setting up hooks...
✓ Hooks registered

3. Forward pass with gradients...
⚠ Caught view/inplace error, trying alternative method...

TRYING ALTERNATIVE GRAD-CAM METHOD
✓ Alternative CAM generated (activation-based)

✓ Grad-CAM successfully generated!
Heatmap shape: (20, 20)
Heatmap range: [0.263, 1.000]

Creating Explainable AI visualization...
✓ Heatmap created

✓ Saved visualization to: C:\Research\Thermal-Dataset\ExplainableAI\weights\yolov5_lar\xai_output\weapon115_xai.png
✓ Image 1/20 complete!

IMAGE 2/20: weapon20.jpg
Loading image...
✓ Image loaded: (640, 640, 3)
✓ Image resized: (640, 640, 3)
Conver